# PropTech 360 | Data Engineering Graduation Capstone
### Industry: Commercial real estate and property management · Level: Intermediate · Total: 100 points

**Student name:** ____________________  **Cohort:** _______APRIL________  **Submission date:** ______OCT 10, 2026_______

> **Assessment notebook, not a worked solution.** Execute the dataset generator, complete the `TODO` tasks independently, and provide evidence of local and AWS runs. Use fictional, synthetic records only.

**Scenario.** MetroSpace Property Group manages a portfolio of commercial buildings. Property, unit, lease, repair, and energy records arrive as disconnected CSV exports. Management cannot reliably compare occupancy, earned rental revenue, maintenance SLA performance, or energy use across buildings. You have been hired to design a reproducible analytics pipeline and explain its reliability and operating cost.

**Mission.** Ingest raw source files; detect and address quality defects; transform data at scale; model and query a PostgreSQL mart; then land curated outputs in an AWS serverless analytics architecture. Your solution must be rerunnable without silently doubling records.

## 1. Five assessed topics and end-to-end architecture

1. **Python ingestion and file handling** — load five raw CSV extracts and record ingestion metadata.
2. **Pandas ETL and data quality** — profile, validate, quarantine, and clean datasets with explicit quality rules.
3. **PySpark transformations** — derive monthly occupancy, rental revenue, maintenance SLA, and energy metrics.
4. **PostgreSQL, SQL, and analytical modeling** — implement a dimensional mart and solve business questions.
5. **AWS serverless pipeline** — S3 raw/curated zones, Lambda, Glue Data Catalog, Athena, IAM, and CloudWatch.

**Reference architecture**

`CSV source extracts → Python intake / raw zone → Pandas validation + quarantine → PySpark curated Parquet → PostgreSQL dimensional mart → SQL reporting`

`                                                                     ↘ S3 curated → Glue Catalog → Athena`  
`S3 raw → S3 event → Lambda (validation / orchestration metadata) → S3 audit + CloudWatch`

**Design constraint:** Do not run Spark inside Lambda. Lambda handles lightweight event validation or metadata; perform the Spark workload separately, locally or on an appropriately provisioned service. Do not claim that S3 automatically refreshes PostgreSQL.

## 2. Business requirements and KPI definitions

Report results **per property and month (January–June 2026)** and provide portfolio-wide totals where appropriate.

- **Occupancy rate:** occupied units / total active units at the **month end**. A unit is occupied when at least one valid lease has `start_date <= month_end <= end_date` (or `end_date` is missing). Flag conflicting simultaneously active leases for the same unit instead of counting the unit twice.
- **Earned monthly rent (simplified):** sum of contractual `monthly_rent_usd` for leases active on the **last day of that month**. This is an analytical proxy, **not cash collected, prorated rent, or recognized accounting revenue**.
- **Maintenance SLA compliance:** share of resolved work orders whose `response_hours` is within target: CRITICAL ≤ 4 h, HIGH ≤ 12 h, MEDIUM ≤ 48 h, LOW ≤ 72 h. Exclude unresolved orders from the compliance denominator and report their count separately. Define and document any additional exclusions.
- **Monthly energy intensity:** sum of valid `energy_kwh` / total property floor area in m²; state whether your chosen denominator includes all or occupied units. Never divide by zero.

**Stakeholders:** portfolio director, operations manager, finance analyst, and cloud platform team. **Grain discipline:** do not join work-order line items directly to lease line items when computing rent; this can multiply totals.

## 3. Environment and setup

**Local:** Python 3.12 recommended for compatibility; pandas, numpy, pyspark, pyarrow, sqlalchemy, psycopg (or psycopg2), Jupyter; PostgreSQL; Java matching your installed PySpark version. Use VS Code or Jupyter. Match PySpark and Java versions to their official compatibility guidance. A local PostgreSQL database and AWS account with appropriate permissions are required for their respective implementation tasks.

**AWS:** S3, Lambda, Glue Data Catalog (crawler or explicit table definitions), Athena, IAM, and CloudWatch. Use a dedicated learning account or permitted lab. AWS resources can incur charges; set a budget alert, avoid sensitive data, restrict IAM privileges, and delete resources after assessment. **Never embed access keys or passwords in notebook outputs or Git.**

**Suggested repository structure**
```text
proptech360-capstone/
  capstone.ipynb
  requirements.txt
  README.md
  data/raw/              # five generated source files
  data/quarantine/       # rejected records + reasons
  data/curated/          # partitioned Parquet + audit/summary
  src/                   # reusable ingestion, validation and ETL modules
  sql/                   # DDL, upserts, reporting queries
  aws/                   # Lambda handler + IAM policy description + deployment steps
  tests/                 # checks for quality rules and KPI logic
  evidence/              # redacted run screenshots / Athena results
```

**Installation (run in the notebook's Python environment):** `python -m pip install pandas numpy pyspark pyarrow sqlalchemy psycopg[binary] pytest boto3` (or pin compatible versions in `requirements.txt`). The next cell checks common local imports; install packages before running it.

In [10]:
# Environment check: record the versions in your README.
import sys
import importlib.metadata as metadata
print('Python:', sys.version.split()[0])
for package in ['pandas', 'numpy', 'pyspark', 'pyarrow', 'sqlalchemy', 'psycopg', 'boto3']:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: NOT INSTALLED (install before its assessed section)')

Python: 3.14.5
pandas: 2.3.3
numpy: 2.5.3
pyspark: NOT INSTALLED (install before its assessed section)
pyarrow: NOT INSTALLED (install before its assessed section)
sqlalchemy: NOT INSTALLED (install before its assessed section)
psycopg: NOT INSTALLED (install before its assessed section)
boto3: NOT INSTALLED (install before its assessed section)


## 4. Generate the synthetic source extracts

Run the generator **once or rerun with the same seed**; it overwrites the same five source files deterministically. Dates and properties are fictional. Deliberately injected quality issues include repeated work orders, negative meter readings, invalid foreign keys, missing response durations, and duplicate unit identifiers. **Do not edit raw files manually**; preserve them as immutable source inputs and fix issues in your pipeline.

**Source contract**

| File | Grain | Primary identifier | Important fields |
|---|---|---|---|
| `properties.csv` | one property | `property_id` | name, city |
| `units.csv` | one unit | `unit_id` | property_id, floor_area_sqm, active |
| `leases.csv` | one lease | `lease_id` | unit_id, start/end dates, monthly_rent_usd |
| `work_orders.csv` | one work order | `work_order_id` | unit_id, opened/closed dates, priority, response_hours, cost_usd |
| `meter_readings.csv` | unit-month | `reading_id` | unit_id, reading_month, energy_kwh |

The file formats intentionally contain dates as text; you must parse and validate them. The generator is **test fixture setup**, not a graded solution.

In [11]:
from pathlib import Path
import csv
import random
from datetime import date, timedelta

rng = random.Random(3602026)
root = Path('data')
raw = root / 'raw'
raw.mkdir(parents=True, exist_ok=True)
for folder in ['quarantine', 'curated']:
    (root / folder).mkdir(parents=True, exist_ok=True)

def write_csv(name, rows):
    path = raw / name
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f'{path}: {len(rows):,} rows')

cities = ['Lagos', 'Abuja', 'Ibadan', 'Port Harcourt']
properties = [dict(property_id=f'P{i:03}', property_name=f'MetroSpace Tower {i:02}',
                   city=cities[(i-1) % len(cities)]) for i in range(1, 13)]
units = []
for p in properties:
    for j in range(1, 21):
        units.append(dict(unit_id=f"{p['property_id']}-U{j:03}", property_id=p['property_id'],
                          floor_area_sqm=rng.choice([45, 60, 75, 90, 120, 150]), active=1))
leases = []
for i, unit in enumerate(units, start=1):
    if rng.random() < 0.22:  # intentionally vacant unit
        continue
    start = date(2025, rng.randint(1, 12), rng.randint(1, 25))
    end = date(2026, rng.randint(3, 12), rng.randint(1, 28)) if rng.random() < 0.27 else None
    leases.append(dict(lease_id=f'L{i:05}', unit_id=unit['unit_id'],
                       start_date=start.isoformat(), end_date=end.isoformat() if end else '',
                       monthly_rent_usd=rng.choice([650, 800, 975, 1200, 1500, 1800, 2200])))
priorities = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
work_orders = []
for i in range(1, 1501):
    unit = rng.choice(units)
    opened = date(2026, rng.randint(1, 6), rng.randint(1, 28))
    resolved = rng.random() < 0.82
    response = round(rng.uniform(1, 96), 1)
    work_orders.append(dict(work_order_id=f'WO{i:06}', unit_id=unit['unit_id'],
        opened_date=opened.isoformat(), closed_date=(opened + timedelta(days=rng.randint(0, 15))).isoformat() if resolved else '',
        priority=rng.choice(priorities), response_hours=response if resolved else '',
        cost_usd=round(rng.uniform(20, 1250), 2), status='RESOLVED' if resolved else 'OPEN'))
months = [f'2026-{m:02}-01' for m in range(1, 7)]
readings = []
for unit in units:
    for month in months:
        readings.append(dict(reading_id=f'R{len(readings)+1:06}', unit_id=unit['unit_id'],
                             reading_month=month, energy_kwh=round(rng.uniform(100, 1250), 2)))
# Known intentionally dirty records; do not remove these from the raw source.
work_orders.append(work_orders[8].copy())                     # exact duplicate
work_orders[34]['unit_id'] = 'P999-U999'                       # unknown parent
work_orders[52]['response_hours'] = ''                         # resolved with missing response
work_orders[63]['cost_usd'] = '-300'                           # invalid negative repair cost
readings[17]['energy_kwh'] = '-45'                             # invalid negative consumption
readings[34]['unit_id'] = 'P999-U999'                           # unknown unit
units.append(units[4].copy())                                 # duplicate identifier
for name, rows in [('properties.csv', properties), ('units.csv', units), ('leases.csv', leases),
                   ('work_orders.csv', work_orders), ('meter_readings.csv', readings)]:
    write_csv(name, rows)
print('Raw fixture ready. Retain a copy before any transformations.')

data/raw/properties.csv: 12 rows
data/raw/units.csv: 241 rows
data/raw/leases.csv: 198 rows
data/raw/work_orders.csv: 1,501 rows
data/raw/meter_readings.csv: 1,440 rows
Raw fixture ready. Retain a copy before any transformations.


## 5. Topic 1 — Python ingestion and reproducibility (15 points)

**Tasks:** (a) create a reusable loader for the five source files; (b) validate file existence, expected columns, and nonempty inputs; (c) record filename, row count, ingest timestamp UTC, and SHA-256 digest in an audit manifest; (d) make repeated runs safe—raw source files must remain unchanged; (e) expose clean error messages for a missing file or missing required column.

**Evidence:** reusable `src/ingest.py`, audit manifest, and a test showing controlled failure for a nonexistent path. Use `pathlib`, `hashlib`, `datetime`, `csv` and/or pandas. **Do not** use `print()` as your only error handling strategy.

In [12]:
# STUDENT TASK 1: implement the loader; use the generated files under data/raw.
from pathlib import Path
import hashlib
import json
from io import BytesIO
from datetime import datetime, timezone
import pandas as pd
RAW_DIR = Path('data/raw')
EXPECTED_COLUMNS = {
    'properties.csv': {'property_id', 'property_name', 'city'},
    'units.csv': {'unit_id', 'property_id', 'floor_area_sqm', 'active'},
    'leases.csv': {'lease_id', 'unit_id', 'start_date', 'end_date', 'monthly_rent_usd'},
    'work_orders.csv': {'work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status'},
    'meter_readings.csv': {'reading_id', 'unit_id', 'reading_month', 'energy_kwh'},
}

def ingest_source(path: Path, expected_columns: set):
    """Return a DataFrame and manifest entry; raise a descriptive error on bad input."""
    # TODO: validate file and columns; load with stable dtypes; compute hash and UTC timestamp.
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Source file not found: {path}")

    # Use the same bytes for loading and hashing.
    source_bytes = path.read_bytes()
    try:
        dataset = pd.read_csv(
            BytesIO(source_bytes),
            dtype="string",
            keep_default_na=False,
            encoding="utf-8-sig",
        )
    except (
        pd.errors.EmptyDataError,
        pd.errors.ParserError,
        UnicodeDecodeError,
    ) as error:
        raise ValueError(f"{path.name}: empty file or invalid CSV.") from error

    missing = set(expected_columns) - set(dataset.columns)
    if missing:
        raise ValueError(
            f'{path.name}: missing required columns: {", ".join(sorted(missing))}'
        )
    if dataset.empty:
        raise ValueError(f"{path.name}: no data rows found.")

    manifest_entry = {
        "filename": path.name,
        "row_count": len(dataset),
        "ingest_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "sha256": hashlib.sha256(source_bytes).hexdigest(),
    }
    return dataset, manifest_entry

# TODO: iterate through EXPECTED_COLUMNS and populate datasets + ingestion_manifest.
datasets = {}
ingestion_manifest = []
for filename, columns in EXPECTED_COLUMNS.items():
    dataset, entry = ingest_source(RAW_DIR / filename, columns)
    datasets[filename] = dataset
    ingestion_manifest.append(entry)

# TODO: save the manifest to data/curated/ingestion_manifest.json.
manifest_path = Path('data/curated/ingestion_manifest.json')
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(ingestion_manifest, indent=2) + '\n', encoding='utf-8')

properties = datasets['properties.csv']
units = datasets['units.csv']
leases = datasets['leases.csv']
work_orders = datasets['work_orders.csv']
meter_readings = datasets['meter_readings.csv']
units.head()


,unit_id,property_id,floor_area_sqm,active
0,P001-U001,P001,75,1
1,P001-U002,P001,90,1
2,P001-U003,P001,75,1
3,P001-U004,P001,90,1
4,P001-U005,P001,150,1


## 6. Topic 2 — Pandas validation, cleaning, and quarantine (20 points)

**Tasks:** Profile null counts, duplicate identifiers, date parse failures, invalid category values, non-positive areas, negative money/energy, broken foreign keys, invalid chronological order, and overlapping leases. Implement a **documented policy** for each rule: fail the run, quarantine the row, correct deterministically with justification, or retain and flag. Preserve original raw files. Do not silently drop data or fill unknown business values with zero.

**Required validations:** unique property/unit/lease/work-order/reading IDs; valid `property_id`→properties and `unit_id`→units; start ≤ end for dated leases; opened ≤ closed for resolved work orders; `priority` ∈ {CRITICAL,HIGH,MEDIUM,LOW}; valid status; numeric nonnegative `energy_kwh`, `cost_usd`, and `monthly_rent_usd`; positive `floor_area_sqm`; `reading_month` must fall inside Jan–Jun 2026; at most one reading per `(unit_id, reading_month)`.

**Quarantine contract:** produce a file per dataset containing original row, `reason_code`, and `run_id`. Report input, accepted, quarantined, and duplicate counts, with an explicit reconciliation explaining whether exact duplicates are counted as quarantined. For aggregate KPI computation, a work order missing an SLA response can be retained for volume reporting but must not count as SLA-compliant by default. If you choose another defensible treatment, document it.

**Evidence:** `src/quality.py`, `data/quarantine/`, reconciliation CSV/JSON, at least five assertions or unit tests including foreign-key and duplicate checks.

In [14]:
# STUDENT TASK 2: write quality profiling and validation code in this cell or src/quality.py.
# Suggested starter pattern (expand for all five sources):
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import json
import unittest
from src import quality
import importlib

# Load the latest saved code when this notebook cell is rerun.
importlib.reload(quality)


def profile_table(df: pd.DataFrame, identifier: str) -> dict:
    # TODO: return row count, nulls by column, duplicate identifier count, and types.
    return quality.profile_table(df, identifier)


def validate_and_quarantine(datasets: dict, run_id: str):
    # TODO: parse dates and numbers, enforce source and FK rules, reconcile rejects.
    # Return: clean_datasets, quarantine_datasets, quality_summary
    return quality.validate_and_quarantine(datasets, run_id)


# TODO: tests for duplicate work order, unknown unit, negative kWh, missing response and bad date.
tests = unittest.defaultTestLoader.discover('tests', pattern='test_quality.py')
# Refresh the test module too, because notebooks remember earlier imports.
import test_quality
importlib.reload(test_quality)
tests = unittest.defaultTestLoader.loadTestsFromModule(test_quality)
result = unittest.TextTestRunner().run(tests)
assert result.wasSuccessful(), 'Fix the failing tests before saving results.'

# TODO: persist clean records into a staging area, separate from untouched raw data.
run_id = datetime.now(timezone.utc).isoformat()
clean_datasets, quarantine_datasets, quality_summary = validate_and_quarantine(datasets, run_id)

for folder, tables in [('staging', clean_datasets), ('quarantine', quarantine_datasets)]:
    output = Path('data') / folder
    output.mkdir(parents=True, exist_ok=True)
    for filename, table in tables.items():
        table.to_csv(output / filename, index=False)

report = Path('data/curated/quality_summary.json')
report.parent.mkdir(parents=True, exist_ok=True)
report.write_text(json.dumps(quality_summary, indent=2), encoding='utf-8')

# Review how many rows passed and failed in each table.
pd.DataFrame(quality_summary).T[['input', 'accepted', 'quarantined']]


..........
----------------------------------------------------------------------
Ran 10 tests in 0.268s

OK


,input,accepted,quarantined
properties.csv,12,12,0
units.csv,241,240,1
leases.csv,198,198,0
work_orders.csv,1501,1498,3
meter_readings.csv,1440,1438,2


## 7. Topic 3 — PySpark curated transformations (25 points)

**Tasks:** (a) start a SparkSession; load validated staging data using explicit schemas where practical; (b) generate the six calendar month ends; (c) compute a unit-month occupancy and monthly-rent snapshot without duplicate lease multiplication; (d) aggregate maintenance per property-month with compliant and noncompliant **resolved** cases plus open cases; (e) aggregate valid energy by property-month; (f) join the independently aggregated KPI tables on `(property_id, month)`; (g) write partitioned Parquet by `month` to `data/curated/property_month_kpis/`.

**Critical transformations:** use DataFrame joins, groupBy/aggregations, casts, date operations, `when`, and window functions where appropriate. Avoid collecting entire fact tables to the driver. Verify all 12 properties × 6 months = **72 property-month output rows** provided no property has been intentionally excluded. Use `countDistinct` or a pre-aggregated unit-month snapshot to avoid multiplying occupancy and rent during joins.

**Minimum columns:** `property_id`, `month`, `total_units`, `occupied_units`, `occupancy_rate`, `earned_monthly_rent_usd`, `resolved_orders`, `sla_eligible_orders`, `sla_compliant_orders`, `open_orders`, `sla_compliance_rate`, `energy_kwh`, `floor_area_sqm`, `energy_intensity_kwh_sqm`.

**Edge cases:** no leases at month end, all orders unresolved, missing response durations, zero/invalid floor area, and duplicate identifiers. Explain your chosen denominator for each rate. Write a quality assertion for unique `(property_id, month)` and a range check for percentages.

In [ ]:
# STUDENT TASK 3: PySpark startup and transformation starter.
# If imports fail, install/configure the local Spark + Java stack first.
from pyspark.sql import SparkSession, functions as F, Window
spark = (SparkSession.builder
         .appName('PropTech360-Capstone')
         .master('local[*]')
         .getOrCreate())
print('Spark version:', spark.version)

# TODO: read VALIDATED staging inputs, NOT dirty raw files.
# TODO: create a six-month calendar with month-end dates.
# TODO: build unit-month lease snapshot; detect/handle lease overlap.
# TODO: calculate property-month occupancy/rent, SLA, and energy INDEPENDENTLY.
# TODO: join the grouped results, assert exactly 72 unique property-month rows.
# TODO: write Parquet partitioned by month; document output mode and rerun behavior.
# spark.stop()  # run after your transformations, not before.

## 8. Topic 4 — PostgreSQL dimensional mart and analytics (20 points)

**Tasks:** Create a PostgreSQL database `proptech360` and dimensional schema with `dim_property`, `dim_month`, and `fact_property_month`. Document natural keys, surrogate keys if used, primary and foreign keys, NOT NULL constraints, valid-range checks, and indexes. Load the curated property-month dataset into the fact table using parameterized inserts or a bulk loader. Implement an idempotent `INSERT ... ON CONFLICT ... DO UPDATE` strategy at the `(property_id, month)` grain; rerun loading and prove row counts stay stable.

**Write and submit SQL answering:**

1. Rank properties by average monthly occupancy for January–June 2026, displaying numerator and denominator context.
2. Calculate monthly earned contractual rent by city (not payment receipts).
3. Show maintenance SLA compliance by property, including counts of eligible resolved and excluded unresolved/missing-response records.
4. Find property-months with energy intensity above the portfolio's monthly average; show both intensity and total kWh.
5. Create a portfolio dashboard view with occupancy rate, earned rent, SLA compliance, and energy intensity; explain handling of NULL rates.

**Evidence:** `sql/01_schema.sql`, `02_load.sql` or `src/load_postgres.py`, `03_analytics.sql`, and screenshots/text output for five queries. Verify PostgreSQL results reconcile with Spark aggregates. Do not paste database credentials into the notebook.

In [ ]:
# STUDENT TASK 4: Example secure connection pattern; this cell intentionally does not connect.
import os
# Install sqlalchemy + psycopg; set DATABASE_URL in your local environment, not in Git.
DATABASE_URL = os.getenv('DATABASE_URL')
if DATABASE_URL is None:
    print('Set DATABASE_URL privately before running your PostgreSQL load.')
# TODO: implement DDL in sql/01_schema.sql and an idempotent staging + upsert loader.
# TODO: query information_schema / constraints to validate schema; rerun to verify 72 fact rows.
# TODO: answer the five business questions in sql/03_analytics.sql.

## 9. Topic 5 — AWS serverless lake and Athena (15 points)

**Create and document the following pipeline in your permitted AWS environment:**

1. Configure a uniquely named **private S3 bucket** with `raw/`, `curated/`, and `audit/` prefixes. Upload a raw CSV to `raw/`; upload **your actual locally generated curated Parquet** to `curated/property_month_kpis/` (preserving partitions). Avoid public ACLs; use encryption defaults and least-privilege IAM.
2. Create a **Python Lambda** triggered by new objects under `raw/`. The handler checks prefix, extension, nonzero size and relevant object metadata, then records an audit JSON under `audit/` and emits structured logs to **CloudWatch**. Add robust handling for event retries and duplicate notifications (e.g., deterministic audit key based on bucket/key/version or eTag and idempotent overwrite), and **prevent a trigger loop** by only triggering on `raw/`.
3. Register the curated Parquet dataset with the **Glue Data Catalog**, through a crawler or explicit schema, and validate inferred partition columns and types. Configure an **Athena query result S3 location** with permissions, and query the catalogued dataset.
4. Run Athena queries for top five property-month occupancy values and portfolio monthly energy totals. Reconcile the monthly energy totals against your local Spark/PostgreSQL results. Capture CloudWatch Lambda logs and an Athena query result as evidence.
5. Explain IAM permissions, S3 event notification behavior, region consistency, expected costs, operational failure modes, and teardown steps. **Do not** give Lambda write permissions across all S3 buckets.

**Division of responsibility:** Lambda validates new raw arrivals and writes an audit; the **Spark computation is performed separately** and its outputs are explicitly uploaded to curated storage. There is no claim that the Lambda event by itself performs the full local Spark pipeline.

**If a course-lab account blocks resource creation:** submit `aws/lambda_handler.py`, architecture diagram, least-privilege IAM pseudopolicy with resource scoping, AWS CLI/console deployment steps, sample S3 event and local Lambda test output, Glue/Athena DDL or crawler configuration, and a clear list of steps you could not execute. Your instructor should apply the same preannounced evidence policy across the cohort.

In [ ]:
# STUDENT TASK 5: Minimal AWS Lambda handler outline — save the completed version to aws/lambda_handler.py.
# This handler is NOT run automatically by the notebook. Implement, deploy and test independently.
LAMBDA_HANDLER_STARTER = r''''''
import json
import os
import boto3
from urllib.parse import unquote_plus

s3 = boto3.client('s3')
AUDIT_BUCKET = os.environ.get('AUDIT_BUCKET')

def lambda_handler(event, context):
    # TODO: iterate Records safely; validate s3:ObjectCreated event, raw/ prefix and .csv suffix.
    # TODO: decode the object key, inspect metadata with head_object and reject zero-size files.
    # TODO: derive idempotent audit key from bucket/key/object version or eTag.
    # TODO: write JSON audit under audit/ with scoped S3 permissions; structured logs.
    # TODO: handle or report partial failure without swallowing exceptions.
    raise NotImplementedError('Student implementation required')
''''''
print('Lambda starter ready; copy into aws/lambda_handler.py and complete the TODOs.')

## 10. Testing, deliverables, and presentation (5 points)

**Automated validation checklist:** prove the generator can be rerun; raw hashes stay unchanged; key uniqueness and foreign keys hold after cleaning; quarantine reasons reconcile; Spark emits 72 unique property-month rows; occupancy and SLA rates are between 0 and 1 or NULL when denominator is zero; monthly rent is not multiplied by work-order counts; PostgreSQL repeated upsert retains 72 fact rows; Athena energy totals reconcile with local output for a selected month.

**Submit:** completed `.ipynb`; runnable repository with `README.md`, pinned `requirements.txt`, `src/`, `sql/`, `aws/`, `tests/`, generated synthetic raw data, sample quarantine/curated outputs (or reproducible generation instructions), a short architecture diagram, and redacted evidence. In README, document commands, assumptions, quality decisions, AWS resource names **without credentials**, rerun instructions, and cleanup. Provide a **5–8 minute walkthrough**: the business problem, one intentional data defect, local KPI reconciliation, AWS evidence, and one improvement you would make for production.

**Suggested assessment window:** 10–14 days, individual work or clearly attributed group contributions. Code can be modularized outside the notebook; the notebook should remain the readable end-to-end assessment entry point.

## 11. Grading rubric (100 points)

| Topic | Points | Assessor looks for |
|---|---:|---|
| Python ingestion | 15 | Reusable loader (5); schema/error validation (4); audit digest/metadata (4); rerun safety (2) |
| Pandas data quality | 20 | Profiling and rule coverage (8); explicit quarantine + reasons (6); reconciliation (3); tests (3) |
| PySpark ETL | 25 | Correct grain and snapshot logic (8); SLA and energy transformations (7); robust joins, edge cases and assertions (6); partitioned curated output (4) |
| PostgreSQL and SQL | 20 | Schema and constraints (5); idempotent load (5); five accurate business queries (8); Spark/SQL reconciliation (2) |
| AWS serverless | 15 | Scoped S3 and Lambda event/audit (6); Glue/Athena results (5); IAM/CloudWatch, cost, failure handling and teardown (4) |
| Testing, documentation and demo | 5 | Repeatable tests (2); README/architecture (2); clear demo (1) |
| **Total** | **100** | |

**Assessment principles:** Award points for correct, reproducible and evidenced work, not merely screenshots or copied service configurations. A correct alternative design is acceptable when it meets the stated grain, definitions, validation, and security requirements. Record your institution's policy for restricted AWS lab access before grading.

### Final student reflection

Write 200–300 words: Which source defect most threatened KPI correctness? Why is the property-month fact grain useful? What is one drawback of the serverless design? What would you improve for production scale, monitoring, or data governance?